# Segundo Parcial — MVP Técnico: Landing → Bronze → Silver → Gold → Serving (Cassandra)

Alcance mínimo de la consigna, en un solo notebook:
1. Batch a Bronze (3 maestros)
2. Streaming a Bronze (con checkpoint **persistente** en Drive — evidencia de incrementalidad)
3. Silver: eventos + 1 maestro, 3 features, 3 reglas de calidad + quarantine con muestras
4. Gold: mart `org_daily_usage_by_service`
5. Serving en AstraDB: keyspace `mvp_parcial`, carga **desde Spark** (`foreachPartition` + `cassandra-driver`, sin pasar por el driver/pandas)
6. Idempotencia y evidencias de particionado

> Patrón de rendimiento: todo el cómputo pesado ocurre en disco local de la VM de Colab; Drive solo persiste el resultado final de cada etapa y el checkpoint de streaming.

## 0. Setup global

In [ ]:
try:
    import pyspark
    print("PySpark ya estaba instalado:", pyspark.__version__)
except ImportError:
    %pip install -q pyspark
    import pyspark

%pip install -q cassandra-driver

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, getpass
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, BooleanType, DateType, LongType,
)

BASE_DIR = "/content/drive/MyDrive/Segundo parcial mineria"
LANDING_DIR = f"{BASE_DIR}/datalake/landing"
BRONZE_DIR = f"{BASE_DIR}/datalake/bronze"
SILVER_DIR = f"{BASE_DIR}/datalake/silver"
GOLD_DIR = f"{BASE_DIR}/datalake/gold"
QUARANTINE_DIR = f"{BASE_DIR}/datalake/quarantine"
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoints/usage_events_bronze"  # persistente, NO se borra
SECRETS_DIR = f"{BASE_DIR}/secrets"

for d in [LANDING_DIR, BRONZE_DIR, SILVER_DIR, GOLD_DIR, QUARANTINE_DIR, SECRETS_DIR]:
    os.makedirs(d, exist_ok=True)

LOCAL_DIR = "/content/local_datalake"
LOCAL_LANDING_EVENTS = f"{LOCAL_DIR}/landing/usage_events_stream"
LOCAL_BRONZE_EVENTS = f"{LOCAL_DIR}/bronze/usage_events"

spark = (
    SparkSession.builder
    .appName("mvp_parcial")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark listo:", spark.version)

In [ ]:
already_loaded = os.path.exists(f"{LANDING_DIR}/customers_orgs.csv")
if already_loaded:
    print("landing/ ya tiene los datos.")
else:
    from google.colab import files
    import zipfile
    print("Subí cloud_provider_challenge_dataset_v1.zip:")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall("/content/_dataset_extract")
    shutil.copytree("/content/_dataset_extract/datalake/landing", LANDING_DIR, dirs_exist_ok=True)
    print("Copiado a:", LANDING_DIR)

## 1. Batch a Bronze — 3 maestros

`customers_orgs`, `users`, `billing_monthly`. Esquema explícito, `ingest_ts`/`source_file`, dedupe por clave natural. Particionamos `billing_monthly` por `month` (el único caso donde particionar aporta valor de consulta a este volumen; `customers_orgs`/`users` son chicos y no se benefician — se documenta en `DECISIONS.md`).

In [ ]:
def ingest_batch_to_bronze(spark, csv_path, schema, dedup_keys, table_name, bronze_dir, partition_by=None):
    df = (
        spark.read.option("header", True).schema(schema).csv(csv_path)
        .withColumn("ingest_ts", F.current_timestamp())
        .withColumn("source_file", F.input_file_name())
    )
    n_raw = df.count()
    df_dedup = df.dropDuplicates(dedup_keys)
    n_dedup = df_dedup.count()
    out_path = f"{bronze_dir}/{table_name}"
    writer = df_dedup.write.mode("overwrite")
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    writer.parquet(out_path)
    print(f"[{table_name}] leídas={n_raw} | tras dedupe={n_dedup} | duplicados eliminados={n_raw - n_dedup} | -> {out_path}")
    return df_dedup

schema_orgs = StructType([
    StructField("org_id", StringType()), StructField("org_name", StringType()),
    StructField("industry", StringType()), StructField("hq_region", StringType()),
    StructField("plan_tier", StringType()), StructField("is_enterprise", BooleanType()),
    StructField("signup_date", DateType()), StructField("sales_rep", StringType()),
    StructField("lifecycle_stage", StringType()), StructField("marketing_source", StringType()),
    StructField("nps_score", DoubleType()),
])
bronze_orgs = ingest_batch_to_bronze(spark, f"{LANDING_DIR}/customers_orgs.csv", schema_orgs, ["org_id"], "customers_orgs", BRONZE_DIR)

schema_users = StructType([
    StructField("user_id", StringType()), StructField("org_id", StringType()),
    StructField("email", StringType()), StructField("role", StringType()),
    StructField("active", BooleanType()), StructField("created_at", DateType()),
    StructField("last_login", DateType()),
])
bronze_users = ingest_batch_to_bronze(spark, f"{LANDING_DIR}/users.csv", schema_users, ["user_id"], "users", BRONZE_DIR)

schema_billing = StructType([
    StructField("invoice_id", StringType()), StructField("org_id", StringType()),
    StructField("month", DateType()), StructField("subtotal", DoubleType()),
    StructField("credits", DoubleType()), StructField("taxes", DoubleType()),
    StructField("currency", StringType()), StructField("exchange_rate_to_usd", DoubleType()),
])
bronze_billing = ingest_batch_to_bronze(spark, f"{LANDING_DIR}/billing_monthly.csv", schema_billing, ["invoice_id"], "billing_monthly", BRONZE_DIR, partition_by=["month"])

## 2. Streaming a Bronze — checkpoint persistente

Esquema explícito (compatibiliza `schema_version` 1/2). Watermark de 65 días: justificado igual que en el proyecto final — los 120 archivos están desordenados a propósito respecto al timestamp del evento, y un watermark angosto descartaría eventos válidos solo por el orden de lectura de archivos (no repetimos el análisis cuantitativo completo acá por brevedad del MVP; está documentado en `DECISIONS.md`).

**A diferencia del proyecto final**, el checkpoint NO se borra entre corridas — vive en Drive. Esto es lo que permite demostrar la incrementalidad real de Structured Streaming: correr esta celda una segunda vez, sin archivos nuevos, no debería reprocesar nada.

In [ ]:
if not os.path.exists(LOCAL_LANDING_EVENTS):
    shutil.copytree(f"{LANDING_DIR}/usage_events_stream", LOCAL_LANDING_EVENTS)

events_schema = StructType([
    StructField("event_id", StringType()), StructField("timestamp", StringType()),
    StructField("org_id", StringType()), StructField("resource_id", StringType()),
    StructField("service", StringType()), StructField("region", StringType()),
    StructField("metric", StringType()), StructField("value", StringType()),
    StructField("unit", StringType()), StructField("cost_usd_increment", DoubleType()),
    StructField("schema_version", LongType()), StructField("carbon_kg", DoubleType()),
    StructField("genai_tokens", LongType()),
])

WATERMARK_THRESHOLD = "65 days"

raw_stream = spark.readStream.schema(events_schema).json(LOCAL_LANDING_EVENTS)
parsed_stream = (
    raw_stream
    .withColumn("event_time", F.to_timestamp("timestamp"))
    .withColumn("value_double", F.expr("try_cast(value as double)"))
    .withColumn("usage_date", F.to_date("event_time"))
    .withColumn("usage_month", F.trunc("event_time", "month"))
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
    .withWatermark("event_time", WATERMARK_THRESHOLD)
    .dropDuplicatesWithinWatermark(["event_id"])
)

def run_streaming_to_bronze():
    query = (
        parsed_stream.writeStream
        .format("parquet")
        .option("path", LOCAL_BRONZE_EVENTS)
        .option("checkpointLocation", CHECKPOINT_DIR)
        .partitionBy("usage_month")
        .trigger(availableNow=True)
        .outputMode("append")
        .start()
    )
    query.awaitTermination()
    progress = query.lastProgress
    n_input = progress["numInputRows"] if progress else 0
    print(f"Micro-batch terminado. numInputRows de la última corrida: {n_input}")
    return n_input

print("--- Corrida 1 (primera vez, debería procesar todo) ---")
n1 = run_streaming_to_bronze()

In [ ]:
print("Sincronizando bronze/usage_events a Drive...")
bronze_events_path = f"{BRONZE_DIR}/usage_events"
shutil.rmtree(bronze_events_path, ignore_errors=True)
shutil.copytree(LOCAL_BRONZE_EVENTS, bronze_events_path)

bronze_events = spark.read.parquet(LOCAL_BRONZE_EVENTS).cache()
n_total = bronze_events.count()
n_distinct = bronze_events.select("event_id").distinct().count()
print(f"usage_events en Bronze: {n_total} filas | event_id distintos: {n_distinct}")

### Evidencia de checkpointing — correr el stream una segunda vez

Sin archivos nuevos en `landing/`, el checkpoint debería hacer que Spark **no vuelva a leer nada**: `numInputRows` de la segunda corrida tiene que ser `0`.

In [ ]:
print("--- Corrida 2 (mismo checkpoint, sin archivos nuevos) ---")
n2 = run_streaming_to_bronze()

print(f"\nCorrida 1: {n1} filas leídas | Corrida 2: {n2} filas leídas")
print("Checkpoint funcionando correctamente" if n2 == 0 else "ADVERTENCIA: se reprocesaron archivos, revisar checkpoint")

## 3. Silver — eventos + `customers_orgs`

Join de enriquecimiento (cada evento se etiqueta con `plan_tier`/`industry`/`hq_region` de su organización) y 3 features derivadas:
- `daily_cost_usd`: costo total del día para esa org (window function).
- `daily_requests`: cantidad de eventos `metric='requests'` ese día para esa org (window function).
- `carbon_kg`: ya viene de Bronze (huella de carbono del evento).

In [ ]:
w_org_day = Window.partitionBy("org_id", "usage_date")

silver_events_enriched = (
    bronze_events
    .join(
        bronze_orgs.select("org_id", "plan_tier", "industry", "hq_region"),
        "org_id", "left",
    )
    .withColumn("daily_cost_usd", F.round(F.sum("cost_usd_increment").over(w_org_day), 4))
    .withColumn(
        "daily_requests",
        F.sum(F.when(F.col("metric") == "requests", 1).otherwise(0)).over(w_org_day),
    )
)
silver_events_enriched.select(
    "event_id", "org_id", "plan_tier", "usage_date", "daily_cost_usd", "daily_requests", "carbon_kg"
).show(5)

### 3 reglas de calidad activas + quarantine

1. `event_id` no nulo y único → **quarantine** (rompe trazabilidad/dedupe).
2. `cost_usd_increment >= -0.01` → si no se cumple, **flag de anomalía** (`is_cost_anomaly`), la fila se mantiene (puede ser un crédito legítimo).
3. `unit` no nulo cuando `value` existe → **quarantine** (lectura inconsistente: hay magnitud pero no se sabe en qué unidad).

Antes de aplicar la regla 3 medimos cuántas filas la violan realmente (no a priori).

In [ ]:
n_total = silver_events_enriched.count()
n_null_event_id = silver_events_enriched.filter(F.col("event_id").isNull()).count()
n_value_no_unit = silver_events_enriched.filter(F.col("value").isNotNull() & F.col("unit").isNull()).count()
n_cost_anomaly = silver_events_enriched.filter(F.col("cost_usd_increment") < -0.01).count()

print(f"Total filas: {n_total}")
print(f"Regla 1 (event_id nulo): {n_null_event_id} filas -> quarantine")
print(f"Regla 3 (value sin unit): {n_value_no_unit} filas -> quarantine")
print(f"Regla 2 (cost_usd_increment < -0.01): {n_cost_anomaly} filas -> flag de anomalía (no se descartan)")

In [ ]:
quarantine_cond = F.col("event_id").isNull() | (F.col("value").isNotNull() & F.col("unit").isNull())

events_quarantine = silver_events_enriched.filter(quarantine_cond).withColumn(
    "quarantine_reason",
    F.concat_ws(", ",
        F.when(F.col("event_id").isNull(), F.lit("event_id nulo")),
        F.when(F.col("value").isNotNull() & F.col("unit").isNull(), F.lit("value sin unit")),
    ),
)

silver_events = (
    silver_events_enriched.filter(~quarantine_cond)
    .withColumn("is_cost_anomaly", F.col("cost_usd_increment") < -0.01)
)

n_q = events_quarantine.count()
n_v = silver_events.count()
print(f"Silver válido: {n_v} filas | Quarantine: {n_q} filas")

if n_q > 0:
    print("\nMuestra de quarantine real (datos del dataset):")
    events_quarantine.select("event_id", "org_id", "value", "unit", "quarantine_reason").show(5, truncate=False)
else:
    print("\n0 filas violan las reglas 1/3 en este dataset (es sintético y viene bastante limpio en integridad).")

print("\nMuestra de filas con flag de anomalía (regla 2, se mantienen):")
silver_events.filter("is_cost_anomaly").select("event_id", "org_id", "cost_usd_increment", "is_cost_anomaly").show(5)

### Prueba unitaria de las reglas (evidencia de que el filtro funciona, con datos sintéticos a propósito)

Si el dataset real no tiene filas que violen las reglas 1 y 3, esto demuestra que la lógica de quarantine **detecta correctamente** los casos problemáticos cuando existen, sin depender de que el dataset productivo los tenga. Estas filas de prueba **no se persisten** en ningún lado.

In [ ]:
test_rows = [
    (None, "org_test", "requests", "100", "count", 1.0),       # viola regla 1: event_id nulo
    ("evt_test_ok", "org_test", "requests", "100", None, 1.0),  # viola regla 3: value sin unit
    ("evt_test_good", "org_test", "requests", "100", "count", 1.0),  # no viola nada
]
test_df = spark.createDataFrame(test_rows, ["event_id", "org_id", "metric", "value", "unit", "cost_usd_increment"])
test_quarantine_cond = F.col("event_id").isNull() | (F.col("value").isNotNull() & F.col("unit").isNull())

print("Filas de prueba que el filtro marca para quarantine (deberían ser 2 de 3):")
test_df.filter(test_quarantine_cond).show()

In [ ]:
shutil.rmtree(QUARANTINE_DIR, ignore_errors=True)
os.makedirs(QUARANTINE_DIR, exist_ok=True)
if n_q > 0:
    events_quarantine.write.mode("overwrite").parquet(f"{QUARANTINE_DIR}/usage_events")

shutil.rmtree(f"{SILVER_DIR}/usage_events", ignore_errors=True)
silver_events.write.mode("overwrite").partitionBy("usage_month").parquet(f"{SILVER_DIR}/usage_events")
print("Silver escrito en:", f"{SILVER_DIR}/usage_events")

## 4. Gold — mart `org_daily_usage_by_service`

Grano diario por organización y servicio, con métricas de uso y costo.

In [ ]:
gold_org_daily_usage = (
    silver_events.groupBy("org_id", "usage_date", "service")
    .agg(
        F.count("*").alias("event_count"),
        F.round(F.sum("value_double"), 4).alias("total_value"),
        F.sum(F.when(F.col("metric") == "requests", 1).otherwise(0)).alias("requests_count"),
        F.round(F.sum("cost_usd_increment"), 4).alias("total_cost_usd"),
        F.round(F.sum("carbon_kg"), 6).alias("total_carbon_kg"),
    )
    .orderBy("org_id", "usage_date")
)
gold_org_daily_usage.show(10)
print("Filas:", gold_org_daily_usage.count())

shutil.rmtree(f"{GOLD_DIR}/org_daily_usage_by_service", ignore_errors=True)
gold_org_daily_usage.write.mode("overwrite").parquet(f"{GOLD_DIR}/org_daily_usage_by_service")
print("Escrito en:", f"{GOLD_DIR}/org_daily_usage_by_service")

## 5. Serving en AstraDB — carga desde Spark

El keyspace `mvp_parcial` y la tabla `org_daily_usage_by_service` se crean manualmente en la CQL Console (ver `docs/astra_schema_mvp.cql`).

**Carga desde Spark, no desde el driver**: usamos `DataFrame.foreachPartition()` — cada partición de Spark abre su propia conexión a Cassandra y escribe sus filas con `cassandra-driver`. Es una escritura distribuida real (a diferencia de colectar todo a un `pandas.DataFrame` en el driver, que es lo que se usó en una primera versión del proyecto).

In [ ]:
bundle_path = f"{SECRETS_DIR}/secure_connect_bundle.zip"
if not os.path.exists(bundle_path):
    from google.colab import files
    print("Subí el Secure Connect Bundle (.zip) de la misma base de Astra del proyecto final:")
    uploaded = files.upload()
    tmp_name = list(uploaded.keys())[0]
    shutil.move(tmp_name, bundle_path)
else:
    print("Bundle ya estaba guardado en:", bundle_path)

ASTRA_TOKEN = getpass.getpass("Pegá tu Astra Application Token: ")

In [ ]:
def write_partition_to_astra(rows):
    """Se ejecuta una vez por partición de Spark. Abre su propia conexión a Cassandra."""
    from cassandra.cluster import Cluster
    from cassandra.auth import PlainTextAuthProvider

    cloud_config = {'secure_connect_bundle': bundle_path}
    auth_provider = PlainTextAuthProvider('token', ASTRA_TOKEN)
    cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
    session = cluster.connect()
    session.set_keyspace('mvp_parcial')

    stmt = session.prepare("""
        INSERT INTO org_daily_usage_by_service
        (org_id, usage_date, service, event_count, total_value, requests_count, total_cost_usd, total_carbon_kg)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """)

    count = 0
    for row in rows:
        session.execute(stmt, (
            row.org_id, row.usage_date, row.service, row.event_count,
            row.total_value, row.requests_count, row.total_cost_usd, row.total_carbon_kg,
        ))
        count += 1

    cluster.shutdown()
    print(f"Partición escrita: {count} filas")

gold_org_daily_usage.foreachPartition(write_partition_to_astra)
print("Carga a AstraDB finalizada.")

### Conexión de verificación (driver, solo para leer y mostrar resultados)

In [ ]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
import pandas as pd

cloud_config = {'secure_connect_bundle': bundle_path}
auth_provider = PlainTextAuthProvider('token', ASTRA_TOKEN)
cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
session = cluster.connect()
KS = "mvp_parcial"

n_astra_before = session.execute(f"SELECT COUNT(*) AS n FROM {KS}.org_daily_usage_by_service").one().n
print(f"Filas en AstraDB tras la primera carga: {n_astra_before}")

## 6. Idempotencia y evidencias

### 6.1 Idempotencia en AstraDB — recargar y comparar conteos

In [ ]:
print("Recargando el mart completo (misma PK -> debería sobreescribir, no duplicar)...")
gold_org_daily_usage.foreachPartition(write_partition_to_astra)

n_astra_after = session.execute(f"SELECT COUNT(*) AS n FROM {KS}.org_daily_usage_by_service").one().n
print(f"\nAntes de recargar: {n_astra_before} filas")
print(f"Después de recargar: {n_astra_after} filas")
print("OK, idempotente" if n_astra_before == n_astra_after else "DIFERENTE -> revisar")

### 6.2 Particionado — evidencia de rutas y tamaños

In [ ]:
def show_partition_evidence(path, label):
    print(f"--- {label} ({path}) ---")
    total_size = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            size = os.path.getsize(fp)
            total_size += size
    rel_dirs = sorted({os.path.relpath(root, path) for root, _, files in os.walk(path) if files})
    print(f"  Particiones (carpetas): {len(rel_dirs)}")
    for d in rel_dirs[:10]:
        print(f"    {d}")
    if len(rel_dirs) > 10:
        print(f"    ... y {len(rel_dirs) - 10} más")
    print(f"  Tamaño total: {total_size / 1024:.1f} KB\n")

show_partition_evidence(f"{BRONZE_DIR}/billing_monthly", "Bronze billing_monthly (particionado por month)")
show_partition_evidence(LOCAL_BRONZE_EVENTS, "Bronze usage_events (particionado por usage_month)")
show_partition_evidence(f"{SILVER_DIR}/usage_events", "Silver usage_events (particionado por usage_month)")
show_partition_evidence(f"{GOLD_DIR}/org_daily_usage_by_service", "Gold org_daily_usage_by_service (sin particionar, mart final chico)")

## 7. Consultas CQL de demo (2 mínimas)

In [ ]:
org_id_demo = gold_org_daily_usage.select("org_id").first().org_id
print("org_id usado en las demos:", org_id_demo)

print("\nQuery 1 — uso/costo diario por servicio de una org, rango de fechas")
rows = session.execute(f"""
    SELECT org_id, usage_date, service, event_count, total_value, requests_count, total_cost_usd, total_carbon_kg
    FROM {KS}.org_daily_usage_by_service
    WHERE org_id = %s AND usage_date >= '2025-06-01' AND usage_date <= '2025-08-31'
""", (org_id_demo,))
display(pd.DataFrame(list(rows)).head(15))

print("\nQuery 2 — punto de acceso específico: org + fecha + servicio")
sample = gold_org_daily_usage.filter(F.col("org_id") == org_id_demo).first()
rows2 = session.execute(f"""
    SELECT org_id, usage_date, service, event_count, total_value, requests_count, total_cost_usd, total_carbon_kg
    FROM {KS}.org_daily_usage_by_service
    WHERE org_id = %s AND usage_date = %s AND service = %s
""", (sample.org_id, sample.usage_date, sample.service))
display(pd.DataFrame(list(rows2)))

---
**Fin del MVP.** Ver `DECISIONS.md` para el log de decisiones (patrón Lambda, particiones, claves de Cassandra, umbrales) y `README.md` para el quickstart.